In [1]:
# Saving region level OSDMA8 for all ensemble members

In [1]:
import xarray as xr
import numpy as np

In [2]:
# === Latitude weighting mean ===
def weighted_mean(da):
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    new_da = da.weighted(weights).mean(("lon", "lat"))
    return new_da

In [ ]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"

region_mask = xr.open_dataarray(f"{MASK_DIR}GBD_Region_Masks_0.10_popgrid.nc")

In [36]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    ensembles = []
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        da = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        regions = []
        # Loop over regions and take OSDMA8 mean for each region
        for i in range(len(region_mask["region"])):
            mask = region_mask.isel(region=i)
            o3_region = weighted_mean(xr.where(mask == 1, da, np.nan))  # OSDMA8 of region
            regions.append(o3_region.drop_vars("region", errors='ignore'))
        osdma8_region = xr.concat(regions, dim=xr.DataArray(region_mask["region"], dims="region", name="region"))

        ensembles.append(osdma8_region)

    region_ens = xr.concat(ensembles, dim=xr.DataArray(np.arange(1, len(ensembles)+1), dims="ensemble", name="ensemble"))

    print(f"Saving region level OSDMA8 to {OSDMA8_DIR}")
    region_ens.to_netcdf(f"{OSDMA8_DIR}OSDMA8_BC_Regional_mean_CESM2_{scenario}_{dates}.nc")

Processing ARISE, Ensemble 01
Processing ARISE, Ensemble 02
Processing ARISE, Ensemble 03
Processing ARISE, Ensemble 04
Processing ARISE, Ensemble 05
Processing ARISE, Ensemble 06
Processing ARISE, Ensemble 10
Saving regional mortality to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/
Processing SSP245, Ensemble 01
Processing SSP245, Ensemble 02
Processing SSP245, Ensemble 03
Processing SSP245, Ensemble 04
Processing SSP245, Ensemble 05
Processing SSP245, Ensemble 06
Processing SSP245, Ensemble 07
Processing SSP245, Ensemble 08
Processing SSP245, Ensemble 09
Processing SSP245, Ensemble 10
Saving regional mortality to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/
